This code restructures the street-location-images dataset (https://www.kaggle.com/datasets/killusions/street-location-images/) into a directory structure that is easily digested as a torchvision ImageFolder dataset.

The input is a directory with images (`location_xxxxxx.png`) and corresponding metadata (`location_xxxxxx.json`), which includes a key `country_code` containing the ISO 3166 code of the target country.

The expected output is a directory with subdirectories for the data splits (`train`/`val`/`test`), each of which contains subdirectories corresponding to the class names (country codes), which then contain the corresponding image files. All classes are represented in every split. The relative size of the splits can be adjusted by editing `SPLIT_FRACS` in the second cell.

Only the mapped subset of the data is used, as some classes in the full dataset are too small to split properly. To prevent unnecessary copying, the destination files are hardlinks to the original files.

In [1]:
# Import necessary modules
import json
import os
import shutil
from collections import Counter
from pathlib import Path

import torchvision

In [2]:
# Define paths for later
SOURCE_PATH = Path("./street-location-images/data_mapped")
TARGET_PATH = Path("./street-location-images/preprocessed_mapped")
# Define the data split sizes
SPLIT_FRACS = {"train": 0.8, "val": 0.1, "test": 0.1}
assert sum(SPLIT_FRACS.values()) == 1

In [3]:
# Retrieve the total dataset size
total_num_images = len(next(os.walk(SOURCE_PATH))[2]) // 2 # count all files, there are 2 per image
print(f"There are a total of {total_num_images} images.")

There are a total of 81505 images.


In [ ]:
# As an intermediary step, sort all images into their classes
all_split_dir = TARGET_PATH / "all"
# delete the whole tree first, if it exists, to clean stale data
if all_split_dir.exists():
    shutil.rmtree(all_split_dir)
for i in range(total_num_images):
    file_name_base = f"location_{i:06d}"
    metadata_file = SOURCE_PATH / (file_name_base + ".json")
    with open(metadata_file, "r") as mf:
        country = json.load(mf).get("country_code", "XX")
    img_file_name = file_name_base + ".png"
    src_file = SOURCE_PATH / img_file_name
    dst_file = all_split_dir / country / img_file_name
    dst_file.parent.mkdir(parents=True, exist_ok=True) # make sure the destination directory exists
    os.link(src_file.absolute(), dst_file)

In [5]:
# Verify that the 'all' dataset is processed correctly by torchvision
all_dataset = torchvision.datasets.ImageFolder(all_split_dir)
assert len(all_dataset) == total_num_images

# Count the size of each class
print(f"There are {len(all_dataset.classes)} classes in the dataset.")
target_counter = Counter(all_dataset.targets)
class_sizes = {class_: target_counter[id] for class_, id in all_dataset.class_to_idx.items()}
print("All classes with sizes:", class_sizes)
print(f"Smallest class size: {min(class_sizes.values())} -- Largest class size: {max(class_sizes.values())}")

There are 75 classes in the dataset.
All classes with sizes: {'AE': 289, 'AL': 350, 'AR': 1473, 'AT': 1657, 'AU': 1811, 'BD': 839, 'BE': 1567, 'BG': 1584, 'BO': 533, 'BR': 1716, 'BW': 420, 'CA': 1565, 'CH': 751, 'CL': 1779, 'CO': 1527, 'CZ': 1681, 'DE': 1717, 'DK': 596, 'DO': 197, 'EC': 677, 'EE': 307, 'ES': 1669, 'FI': 1505, 'FR': 1745, 'GB': 1650, 'GH': 492, 'GR': 1840, 'GT': 537, 'HR': 1525, 'HU': 809, 'ID': 1693, 'IE': 1777, 'IL': 782, 'IN': 1605, 'IT': 1700, 'JP': 1747, 'KE': 734, 'KG': 247, 'KH': 543, 'KR': 818, 'LA': 98, 'LK': 655, 'LS': 537, 'LT': 437, 'LV': 355, 'ME': 230, 'MK': 236, 'MT': 202, 'MX': 1591, 'MY': 1393, 'NG': 839, 'NL': 1632, 'NO': 1496, 'NZ': 1537, 'PE': 1829, 'PH': 1556, 'PL': 1515, 'PT': 1597, 'RO': 1546, 'RS': 669, 'RU': 1739, 'RW': 161, 'SE': 1748, 'SG': 759, 'SI': 287, 'SK': 348, 'SN': 673, 'SZ': 327, 'TH': 1672, 'TR': 1635, 'UA': 1803, 'UG': 215, 'US': 1639, 'UY': 518, 'ZA': 1577}
Smallest class size: 98 -- Largest class size: 1840


In [6]:
# Clean up any existing train/val/test splits first
for split in SPLIT_FRACS:
    split_dir = TARGET_PATH / split
    if split_dir.exists():
        shutil.rmtree(split_dir)
# Split each class into train/val/test
for class_ in all_dataset.classes:
    # Out of all the images in the class ...
    all_class_dir = TARGET_PATH / "all" / class_
    class_files = next(os.walk(all_class_dir))[2]
    # ... calculate how many should belong to each split
    class_train_size = int(SPLIT_FRACS["train"] * len(class_files))
    class_val_size = int(SPLIT_FRACS["val"] * len(class_files))
    # then iterate over all the image files ...
    for i, file_name in enumerate(class_files):
        src_file = all_class_dir / file_name
        # ... determine which split the file should be sorted into ...
        if i < class_train_size:
            split = "train"
        elif class_train_size <= i < (class_train_size + class_val_size):
            split = "val"
        else:
            split = "test"
        # ... and create the link to the source image
        dst_file = TARGET_PATH / split / class_ / file_name
        dst_file.parent.mkdir(parents=True, exist_ok=True) # make sure the destination directory exists
        os.link(src_file.absolute(), dst_file)

In [ ]:
# Verify that the 3 split datasets are loaded correctly
train_dataset = torchvision.datasets.ImageFolder(TARGET_PATH / "train")
val_dataset = torchvision.datasets.ImageFolder(TARGET_PATH / "val")
test_dataset = torchvision.datasets.ImageFolder(TARGET_PATH / "test")

print("The splits have sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

assert len(train_dataset) + len(val_dataset) + len(test_dataset) == total_num_images
assert len(train_dataset.classes) == len(val_dataset.classes) == len(test_dataset.classes) == len(all_dataset.classes)

# The all-split is no longer required and can be deleted (optional)
if all_split_dir.exists():
    shutil.rmtree(all_split_dir)

The splits have sizes: 65174 8111 8220
